# **Hands-On Tutorial on Semi-Numerical Models of Cosmic Reionization**
---
Lecturer: Prof. Tirthankar Roy Choudhury
---
School : Physics of Galaxy Formation, IUCAA Pune
---
Date : June 16, 2026
---

Tutor : Anirban Chakraborty (Email : anirban.chakraborty096@gmail.com)
---

## **GOALS** :  


### To develop familiarity with a popular semi-numerical reionization code -- SCRIPT, and to learn how to use it to generate ionization maps and compute the corresponding 21 cm power spectra for different choices of model parameters.

----

## **Please go through this tutorial by executing the cells sequentially, rather than running all the cells at once!**
---

# **Preliminary : Start a new Google Colab session**

If you have already executed this notebook once and the software installation or working folders are still present on the virtual machine, you must first clear them. To do this, follow these steps:


1.   Go to  ```Runtime``` in the top menu and select the ```Disconnect and delete runtime``` option.

2.   If the session does not reconnect automatically, click the ```Reconnect``` button in the top-right corner of the interface.







##**Part 0 : Download the mock dataset and install all the required packages**

Create a working directory

In [ ]:
import os

# Define the base directory (default working location in Google Colab)
base_path = "/content"

# Create the path for a new working directory called "work"
work_path = os.path.join(base_path, "work")

# Create the directory if it does not already exist
os.makedirs(work_path, exist_ok=True)

# Change the current working directory to the newly created "work" folder
os.chdir(work_path)

# Print and confirm the current working directory
os.getcwd()


Create a folder in the working directory to store all saved plots from the session

In [ ]:
# Create a directory named "saved_figures" in the  working directory
os.makedirs("saved_figures", exist_ok=True)

Download the datafiles from the [Google Drive](https://drive.google.com/drive/folders/1ufCSpuydo_Q7GEGmJotXUvAy-BwdYjWv?usp=drive_link) folder associated with this tutorial

In [ ]:
# Download the simulation snapshot directory from GitHub link
snapshot_dir_path = os.path.join(work_path, "N128L128_2LPT")

if not os.path.exists(snapshot_dir_path):
    print("Snapshot directory not found. Downloading...")
    !gdown --folder https://drive.google.com/drive/folders/1ufCSpuydo_Q7GEGmJotXUvAy-BwdYjWv?usp=drive_link


else:
    print(f"'{snapshot_dir_path}' already exists. Skipping download.")

Let us now import the required python libraries

In [ ]:
# Modules used for interacting with the operating system and Python environment
import os, sys

# Add the current working directory to Python's module search path
sys.path.append(os.getcwd())

# NumPy: library for numerical computations and array operations
import numpy as np

# Matplotlib: library for generating and displaying plots
import matplotlib.pyplot as plt
import matplotlib


In [ ]:
# Define plotting style parameters

plt.rcParams['figure.figsize'] = [12, 8]
plt.rcParams['xtick.labelsize']=18
plt.rcParams['ytick.labelsize']=18
plt.rcParams.update({'font.size': 22})
plt.rcParams.update({'font.family':'serif'})

In [ ]:
def snapNum_to_z(snap_num):
    return 15.0 - 0.5 * snap_num

def z_to_snapNum(z):
    return int(round((15.0 - z) / 0.5))

snap_nums = np.arange(21)
redshifts = snapNum_to_z(snap_nums)

snap_to_redshift = dict(zip(snap_nums, redshifts))
redshift_to_snap = dict(zip(redshifts, snap_nums))

In [ ]:
for snap, z in snap_to_redshift.items():
    print(f"{int(snap):2d}: {float(z):4.1f}")

# A Semi-Numerical Model of Cosmic Reionization  (SCRIPT)

**SCRIPT** stands for **S**emi Numerical **C**ode for **R**eion**I**zation with **P**ho**T**on Conservation. It was introduced to mitigate the issue of photon number non conservation associated with other standard semi-numerical codes based on Excursion Set (ES) methods.

For details, see [Choudhury & Paranjape (2018)](https://doi.org/10.1093/mnras/sty2551) and visit [SCRIPT repository](https://bitbucket.org/rctirthankar/script) for documentation of code  and examples .

The N-body simulation data for this tutorial was generated through the 2LPT code [MUSIC](https://www-n.oca.eu/ohahn/MUSIC/).

---

We will first install **SCRIPT** from the [SCRIPT repository](https://bitbucket.org/rctirthankar/script).

In [ ]:
# Change the current working directory back to the base path
os.chdir(base_path)

# Install SCRIPT and its dependencies

# Install the FFTW3 library (required by SCRIPT for fast Fourier transforms)
!apt-get install libfftw3-dev >/dev/null

# Clone the SCRIPT repository from Bitbucket
!git clone https://bitbucket.org/rctirthankar/script > /dev/null

The above step will create a folder named
```script``` in the base directory. We will need to go inside this folder and then, install the code

In [ ]:
# Move into the SCRIPT directory
script_path = os.path.join(base_path, "script")
os.chdir(script_path)


# Install SCRIPT using pip from the current directory.
try:
    import script
    print("SCRIPT already installed.")
except ImportError:
    !pip install .

    #This will setup various modules required to perform our tasks.

Let us then return to our working directory

In [ ]:
# Go back to the working directory
os.chdir(work_path)

If **SCRIPT** was installed correctly, the next cell should execute without raising any errors.

In [ ]:
# for using the SCRIPT package
import script

## Exercise 1(A): Compute the field : $\zeta \, f_{\mathrm{coll}}(>M_{\mathrm{min}}$) at $z=7$ for $M_{\mathrm{min}}=10^8 M_\odot$ and $\zeta$ = 7




For the snapshot used in this tutorial, the **MUSIC-2LPT** simulation was run with the following specifications:

Boxsize = 128  $h^{-1}$ cMpc

No. of DM particles = $128^3$



Now we will read the binary simulation file generated from **SCRIPT** at $z = 7$.

This file contains infomation about the dark matter particles ($x_i,v_i,M_i$) and box parameters.

For this, we will use the `default_simulation_data` class inside the `script` module.

*Note: You still need to supply the cosmological parameters*




In [ ]:
snapnum_z7=redshift_to_snap[7]
snap_str_z7 = '{:03d}'.format(snapnum_z7)
print("snapfile :", snap_str_z7)

In [ ]:
# Name of the simulation snapshot file (generated by MUSIC)
gadget_snap = os.path.join(work_path,
    "N128L128_2LPT",
    f"snap_{snap_str_z7}"
)
# Directory where SCRIPT will save the generated density and velocity fields
outpath = './script_saved_files'

# Use 1e-3 if the length unit for the MUSIC run was kpc/h, and 1.0 if it was in Mpc/h
scaledist = 1e-3

# Read the binary simulation snapshot and initialize the default simulation data structure
# Cosmological parameters must be supplied here
default_simulation_data = script.default_simulation_data(
    gadget_snap,
    outpath,
    sigma_8=0.829,
    ns=0.961,
    omega_b=0.0482,
    scaledist=scaledist
)

#check if the correct box is picked
print("Redshift of snap (from header): ", default_simulation_data.z)

Once the dark matter particle data is retrieved, the next step is to construct the density, velocity, and halo collapsed fraction fields on a uniform grid, which are required for computing the ionization fields. This can be done using `matter_fields` class inside the `script` module.
  


The number of grids is then simply calculated as : $n_{\mathrm{grid}}$ = $L_{\mathrm{box}}/\Delta x$

**Question:**
If we want to generate the ionization maps at a resolution of $\Delta x$ = 2 $h^{-1}$ cMpc, what should be the value of $n_{\mathrm{grid}}$?





In [ ]:
# Number of grid cells per dimension used to smooth and discretize the density field
ngrid = 32

# Generate (or read) the gridded fields such as density and velocity
matter_fields = script.matter_fields(default_simulation_data, ngrid, outpath, overwrite_files=False)

# If overwrite_files=True, SCRIPT will recompute and overwrite the saved gridded fields.
# If overwrite_files=False, existing gridded fields will be read from disk (if available).

Next, we compute the quantity $ \zeta \, f_{\mathrm{coll}}$ in each cell of the simulation box.

Here $f_{\mathrm{coll}}$ is the fraction of mass inside collapsed objects with mass above a threshold of $M_{\mathrm{min}}$, and $\zeta$ is the effective ionizing efficiency of these sources.

We use the `get_zeta_fcoll` function in the `matter_fields` class.



In [ ]:
# Minimum mass of halo (in solar masses) capable of producing ionizing radiation
log10Mmin = 8.0

# Effective ionizing efficiency of haloes
zeta = 7


## Calculate zeta times collapse fraction for each cell
zeta_times_fcoll_arr = matter_fields.get_zeta_fcoll(zeta,log10Mmin)

print ( 'zeta times fcoll = ' +  '{:.2f}'.format(np.mean(zeta_times_fcoll_arr * (1 + matter_fields.densitycontr_arr))) ) # predicted ionized fraction

In [ ]:
# Compute zeta fcoll (1 + delta_i)
zeta_fcoll_delta_arr = zeta_times_fcoll_arr * (1 + matter_fields.densitycontr_arr)

# ---------------------------------------------------------
# Plot a central slice through the simulation box
# ---------------------------------------------------------

fig, ax = plt.subplots(figsize=(10, 8))

# Position of the slice through the box (in units of box size)
centre = 0.5

# Index corresponding to the chosen slice location
ygrid = int(ngrid * centre)

# Simulation box size (in h^-1 cMpc)
box = default_simulation_data.box

im = ax.imshow(
    zeta_fcoll_delta_arr[:, ygrid, :].T,
    origin='lower',
    extent=[0, box, 0, box],
    interpolation='bicubic',
    cmap=plt.get_cmap('viridis')
)

cbar = plt.colorbar(im, ax=ax)
cbar.set_label(r'$\zeta f_{\rm coll,i}(1+\delta_i)$', fontsize=14)

ax.set_xlabel(r'$x (h^{-1} \, \mathrm{cMpc})$')
ax.set_ylabel(r'$z (h^{-1} \, \mathrm{cMpc})$')
ax.set_title(
    rf'$\zeta f_{{\rm coll,i}}(1+\delta_i)$'
    '\n'
    rf'$\zeta={zeta},\ \log_{{10}}M_{{\min}}={log10Mmin}$'
)

plt.tight_layout()
plt.savefig(
    './saved_figures/SCRIPT_input_zeta' + str(zeta) +
    '_logMmin' + str(log10Mmin) + '.pdf'
)
plt.show()

## Exercise (1B) : Generating ionization maps and plotting power spectra for $\zeta=7$ and $M_{\mathrm{min}}=10^8 M_\odot$

For constructing the ionization map, we need to compute the ionized fraction field. This can be done using the `ionization_map` class. By default, the code uses the **Photon Conservation (PC)** method to generate the ionization maps.



In [ ]:
# Initialize the ionization_map class using the previously computed matter fields
# This object will be used to generate the ionization maps of the simulation box
ionization_map = script.ionization_map(matter_fields)

Once the object is set up, we can compute the **ionized hydrogen fraction (HII)** in each cell of the simulation box using the `get_qi` function of the `ionization_map` class.

In [ ]:
# Compute the ionized hydrogen fraction (q_i) in each cell of the simulation box
qi_arr = ionization_map.get_qi(zeta_times_fcoll_arr)

Let us do a quick check to verify whether the computed ionized hydrogen fraction matches the predicted ionized fraction (a salient feature of SCRIPT)

In [ ]:
qi_mean = np.mean(qi_arr * (1 + matter_fields.densitycontr_arr))
print ( 'mass averaged ionized fraction = ' + '{:.2f}'.format(qi_mean) ) # mass averaged ionized fraction of Hydrogen
print ( 'mean zeta times fcoll = ' +  '{:.2f}'.format(np.mean(zeta_times_fcoll_arr * (1 + matter_fields.densitycontr_arr))) ) # predicted ionized fraction

Once the ionized fraction in each cell is calculated, we construct the **HI overdensity field**, which is given by $\Delta_{\mathrm{HI}}= (1 - x_{\mathrm{HII}}) \,\cdot\, (1 + \delta)$, where $x_{\mathrm{HII}}$ is the ionized hydrogen fraction in each cell and $\delta$ is the density contrast in the same cell.  

Using this HI overdensity field, we can then compute the **HI power spectrum**.


In [ ]:
# Initialize the power spectrum calculation
# Prepare the FFT plans and internal variables

matter_fields.initialize_powspec()

k_edges, k_bins = matter_fields.set_k_edges(nbins=12, kmin=None, kmax=None, log_bins=False)

# Neutral hydrogen fraction field (HI fraction)
x_HI_arr = (1 - qi_arr)

# HI density field:
Delta_HI_arr = (1 - qi_arr) * (1 + matter_fields.densitycontr_arr)

# Compute the binned 21 cm power spectrum
powspec_21cm_binned, kount = ionization_map.get_binned_powspec(
    Delta_HI_arr,
    k_edges=k_edges,
    units='mK',
    convolve=True
)

# k_edges specifies the bin edges in k-space
# units='mK' converts the power spectrum to brightness temperature units
# convolve=True accounts for the finite cell size effectss (CIC window function)

Let us plot the computed 21 cm power spectrum and some two-dimmensional slices from the simulation box

In [ ]:
# Create a square figure to hold multiple panels
fig = plt.figure(figsize=(12,12))

# Top-left panel: matter density field slice
ax_den = fig.add_subplot(221)

# Top-right panel: neutral hydrogen (HI) field slice
ax_HI = fig.add_subplot(222)

# --------------------------------------------------

# Simulation box size (in h^-1 cMpc)
box = default_simulation_data.box

# Position of the slice through the box (in units of box size)
centre = 0.5

# Number of grid cells along each dimension
ngrid = matter_fields.ngrid

# Grid spacing
dx = box / ngrid

# Index corresponding to the chosen slice location
ygrid = int(ngrid * centre)


# Plot the matter overdensity field

# Set color scale limits
vmin = 0
vmax = None

# Display the density field slice
im_den = ax_den.imshow(
    1 + matter_fields.densitycontr_arr[:, ygrid, :].T,
    origin='lower',
    extent=[0, box, 0, box],
    interpolation='bicubic',
    cmap=plt.get_cmap('hot'),
    vmin=vmin,
    vmax=vmax
)

ax_den.set_xlabel(r'$x (h^{-1} \, \mathrm{cMpc})$')
ax_den.set_ylabel(r'$z (h^{-1} \, \mathrm{cMpc})$')
ax_den.set_title(r'$z = ' + '{:.2f}'.format(default_simulation_data.z) + '$')

# Add colorbar for the density field
cbar_den = fig.colorbar(im_den, ax=ax_den, fraction=0.046, pad=0.04)
cbar_den.set_label(r'$\Delta = 1 + \delta$')


# Plot the neutral hydrogen (HI) fraction field

# Set color scale limits for HI fraction
vmin = 0
vmax = 1

# Display HI fraction slice
im_HI = ax_HI.imshow(
    x_HI_arr[:, ygrid, :].T,
    origin='lower',
    extent=[0, box, 0, box],
    interpolation='bicubic',
    cmap=plt.get_cmap('viridis'),
    vmin=vmin,
    vmax=vmax
)


ax_HI.set_xlabel(r'$x (h^{-1} \, \mathrm{cMpc})$')
ax_HI.set_ylabel(r'$z (h^{-1} \, \mathrm{cMpc})$')
ax_HI.set_title(rf'$\ Q_{{\mathrm{{HII}}}}^{{M}} = {qi_mean:.2f}$')

# Add colorbar for HI fraction
cbar_HI = fig.colorbar(im_HI, ax=ax_HI, fraction=0.046, pad=0.04)
cbar_HI.set_label(r'$x_{\mathrm{HI}}$')


# Plot the HI power spectrum

# Bottom panel spanning the full width
ax_pk = fig.add_subplot(212)

# Plot the dimensionless HI power spectrum
ax_pk.plot(
    k_bins,
    k_bins**3 * powspec_21cm_binned / (2 * np.pi**2),
    label='21cm power spectrum',
    marker='o'
)

# Alternative: plot only bins with modes
# ax_pk.plot(kbins_data[kount > 0],
#            kbins_data[kount > 0]**3 * powspec_21cm_binned / (2 * np.pi**2),
#            label='HI power spectrum')


ax_pk.legend(loc='best')

ax_pk.set_xscale('log')
ax_pk.set_yscale('log')
ax_pk.set_xlabel(r'$k ~ [h \, \mathrm{cMpc}^{-1}]$')
ax_pk.set_ylabel(r'$\Delta^2_{21}(k) ~ [\mathrm{mK}^2]$')



fig.tight_layout()
plt.tight_layout()

plt.savefig(
    './saved_figures/SCRIPT_output_zeta' + str(zeta) +
    '_logMmin' + str(log10Mmin) + '.pdf'
)

# Display the figure
plt.show()

Let us now define a Python function that computes the **theoretical 21 cm power spectrum** from SCRIPT for a given set of parameters.

The function will take two parameters:

- $\log_{10}(M_{\mathrm{min}})$ — the minimum halo mass (in solar masses) capable of hosting ionizing sources.
- $\zeta$ — the ionizing efficiency parameter.

and return the **dimensionless 21 cm power spectrum** and the **mass-averaged ionized fraction**.

This will come handy as we go along

In [ ]:
def SCRIPT_model(log10Mmin, zeta, z=7, ngrid=32):
    """
    Generate the ionization and neutral hydrogen fields for a given
    reionization model and compute the corresponding theoretical
    21 cm power spectrum.

    Parameters
    ----------
    log10Mmin : float
        Base-10 logarithm of the minimum halo mass, M_min (in Msun),
        capable of hosting ionizing sources.

    zeta : float
        Effective ionizing efficiency parameter that relates the
        collapsed fraction to the number of ionizing photons produced.

    z : float, optional
        Redshift at which the matter, ionization, and 21 cm fields are
        evaluated. Default is z = 7.

    ngrid : int, optional
        Number of grid cells along each dimension of the simulation box.
        The density, ionization, and neutral hydrogen fields therefore
        contain ngrid^3 cells. Changing ngrid changes the spatial
        resolution, Fourier grid, and Nyquist wavenumber.
        Default is ngrid = 32.

    Returns
    -------
    k_bins : ndarray
        Centres of the spherically averaged wavenumber bins, in units of
        h cMpc^-1, corresponding to the computed 21 cm power spectrum.

    Delta_sq_21cm : ndarray
        Dimensionless 21 cm power spectrum, in units of mK².

    QHII_mean : float
        Mass-averaged ionized fraction,
            <Q_HII> = <q_i (1 + δ)>.

    Delta_HI_arr : ndarray
        Three-dimensional neutral hydrogen overdensity field,
            Δ_HI = (1 - q_i)(1 + δ).

    qi_arr : ndarray
        Three-dimensional ionization field containing the ionized
        fraction, q_i, in each grid cell.

    fcoll_mass_mean : float
        Mass-averaged collapsed fraction,
            <f_coll> = <f_coll (1 + δ)>.
    """

    print(
        f"log10Mmin={log10Mmin}, "
        f"zeta={zeta}, "
        f"z={z}, "
        f"ngrid={ngrid}"
    )

    # Name of the simulation snapshot file (generated by MUSIC)
    z = redshift_to_snap[z]
    snap_str_z = '{:03d}'.format(z)
    print("snapfile :", snap_str_z)

    gadget_snap = os.path.join(work_path,
        "N128L128_2LPT",
        f"snap_{snap_str_z}"
    )


    # Read the binary simulation snapshot and initialize the default simulation data structure
    # Cosmological parameters must be supplied here
    default_simulation_data = script.default_simulation_data(
        gadget_snap,
        outpath='./script_saved_files',
        sigma_8=0.829,
        ns=0.961,
        omega_b=0.0482,
        scaledist=1e-3
    )

    # Generate (or read) the gridded fields such as density and velocity
    matter_fields = script.matter_fields(default_simulation_data, ngrid, outpath, overwrite_files=False)

    # zeta times collapsed fraction field
    zetafcoll_arr = matter_fields.get_zeta_fcoll(zeta, log10Mmin)

    fcoll_arr = matter_fields.get_zeta_fcoll(1, log10Mmin)

    fcoll_mass_mean = np.mean(fcoll_arr * (1 + matter_fields.densitycontr_arr))

    # This object will be used to generate the ionization maps of the simulation box
    ionization_map = script.ionization_map(matter_fields)

    # Ionization field
    qi_arr = ionization_map.get_qi(zetafcoll_arr)

    # Neutral hydrogen overdensity field
    Delta_HI_arr = (1 - qi_arr) * (1 + matter_fields.densitycontr_arr)

    # Mass-averaged ionized fraction
    QHII_mean = np.mean(
        qi_arr * (1 + matter_fields.densitycontr_arr)
    )

    # Power spectrum calculation
    matter_fields.initialize_powspec()

    k_edges, k_bins = matter_fields.set_k_edges(nbins=12, kmin=None, kmax=None, log_bins=False)


    powspec_21cm, kount = ionization_map.get_binned_powspec(
        Delta_HI_arr,
        k_edges=k_edges,
        units='mK',
        convolve=True
    )

    # Dimensionless power spectrum
    Delta_sq_21cm = (
        k_bins**3 * powspec_21cm / (2 * np.pi**2)
    )

    return k_bins, Delta_sq_21cm, QHII_mean, Delta_HI_arr, qi_arr, fcoll_mass_mean

## **Exercise 1** :

### a. Generate ionization maps and 21 cm power spectra for $\log_{10} (M_{\mathrm{min}}/M_\odot)=[8.0, 8.5, 9.0] $ with $\zeta$ = 7.

### b. Compare the power spectra and the mass-averaged ionized fraction for different values of $M_{\mathrm{min}}$.

### c. What do you see from a comparison of power spectrum?

In [ ]:
log10Mmin_arr = np.array([8.0, 8.5, 9.0])
num_Mmin = len(log10Mmin_arr)
zeta_value = 7.0

fig, axes = plt.subplots(
    1, 4,
    figsize=(20, 5)
)

ax_ps = axes[0]

slice_index = matter_fields.ngrid // 2

for loopidx, log10Mmin_value in enumerate(log10Mmin_arr):

    model_kbins, model_Delta_sq_21cm, model_QHII_mean, model_Delta_HI_arr, model_qi_arr, model_fcoll_mass_mean = SCRIPT_model(log10Mmin_value, zeta_value)

    print('mass averaged ionized fraction = '
        + '{:.2f}'.format(model_QHII_mean)
    )

    # ---------------------------------------
    # Power spectrum panel
    # ---------------------------------------

    ax_ps.plot(
        model_kbins,
        model_Delta_sq_21cm,
        lw=2,
        label=rf'$\log_{{10}} M_{{\rm min}}={log10Mmin_value:.1f}$'
    )

    # ---------------------------------------
    # Delta_HI map slice
    # ---------------------------------------

    ax = axes[loopidx + 1]

    im = ax.imshow(
        model_Delta_HI_arr[:, :, slice_index],
        origin='lower',
        vmin=0,
        vmax=1,
        interpolation='bicubic',
    )

    ax.set_title(
        rf'$\log_{{10}} M_{{\rm min}}={log10Mmin_value:.1f}$' '\n'
        rf'$\langle Q^M_{{\rm HII}}\rangle={model_QHII_mean:.2f}$'
    )

    ax.set_xticks([])
    ax.set_yticks([])

    print("-------")

# ---------------------------------------
# Format power spectrum panel
# ---------------------------------------

ax_ps.set_xscale('log')
ax_ps.set_yscale('log')

ax_ps.set_xlabel(
    r'$k~[h\,{\rm cMpc}^{-1}]$'
)

ax_ps.set_ylabel(
    r'$\Delta^2_{21}(k)\,[{\rm mK}^2]$'
)

ax_ps.set_title(
    rf'$\zeta={zeta_value:.1f}$'
)

ax_ps.legend(fontsize=12)

# ---------------------------------------
# Shared colorbar for slices
# ---------------------------------------

#cbar = fig.colorbar(im, ax=axes[1:], shrink=0.8)

#cbar.set_label(r'$Q_{\rm HII}$')

plt.tight_layout()

# Leave room at the bottom
fig.subplots_adjust(bottom=0.18)

# [left, bottom, width, height]
cax = fig.add_axes([0.35, 0.08, 0.45, 0.03])

cbar = fig.colorbar(
    im,
    cax=cax,
    orientation='horizontal'
)

cbar.set_label(r'$\Delta_{\rm HI} = x_{\rm HI}(1+\delta)$')

plt.savefig(
    './saved_figures/SCRIPT_output_vary_logMmin_maps.pdf'
)

plt.show()

## **Exercise 2** :

### a. Generate ionization maps and 21 cm power spectra for $\log_{10} (M_{\mathrm{min}}/M_\odot)=8.0$ with $\zeta$ = [6,8,10].

### b. Compare the power spectra and the mass-averaged ionized fraction for different values of $\zeta$. What do you see from a comparison of power spectrum?

### c. What do you see from a comparison of power spectrum?

In [ ]:
zeta_arr = np.array([6.0, 7.0, 8.0])
log10Mmin_value = 8.0

fig, axes = plt.subplots(1, 4,figsize=(20, 5))

ax_ps = axes[0]

slice_index = matter_fields.ngrid // 2

for loopidx, zeta_value in enumerate(zeta_arr):

    print(
        f"log10Mmin = {log10Mmin_value}, "
        f"zeta = {zeta_value}"
    )

    model_kbins, model_Delta_sq_21cm, model_QHII_mean, model_Delta_HI_arr, model_qi_arr, model_fcoll_mass_mean= \
        SCRIPT_model(
            log10Mmin_value,
            zeta_value
        )

    print(
        'mass averaged ionized fraction = '
        + '{:.2f}'.format(model_QHII_mean)
    )

    # Power spectrum

    ax_ps.plot(
        model_kbins,
        model_Delta_sq_21cm,
        lw=2,
        label=rf'$\zeta={zeta_value:.1f}$'
    )

    # Delta_HI_arr map slice

    ax = axes[loopidx + 1]

    im = ax.imshow(
        model_Delta_HI_arr[:, :, slice_index],
        origin='lower',
        vmin=0,
        vmax=1,
        interpolation='bicubic',
    )

    ax.set_title(
        rf'$\zeta={zeta_value:.1f}$' '\n'
        rf'$\langle Q^M_{{\rm HII}}\rangle={model_QHII_mean:.2f}$'
    )

    ax.set_xticks([])
    ax.set_yticks([])

    print("-------")


ax_ps.set_xscale('log')
ax_ps.set_yscale('log')

ax_ps.set_xlabel(
    r'$k\,[h\,{\rm cMpc}^{-1}]$'
)

ax_ps.set_ylabel(
    r'$\Delta^2_{21}(k)\,[{\rm mK}^2]$'
)

ax_ps.set_title(
    rf'$\log_{{10}}(M_{{\rm min}}/M_\odot)={log10Mmin_value:.1f}$'
)

ax_ps.legend()


plt.tight_layout()

# Leave room at the bottom
fig.subplots_adjust(bottom=0.18)

# [left, bottom, width, height]
cax = fig.add_axes([0.35, 0.08, 0.45, 0.03])

cbar = fig.colorbar(
    im,
    cax=cax,
    orientation='horizontal'
)

cbar.set_label(r'$\Delta_{\rm HI} = x_{\rm HI}(1+\delta)$')

plt.savefig(
    './saved_figures/SCRIPT_output_vary_zeta_maps.pdf'
)

plt.show()

## **Exercise 3** :

### a. Generate ionization maps and 21 cm power spectra for $n_{\rm grid} = [64,32,16]$ with $\log_{10} (M_{\mathrm{min}}/M_\odot)=8.0 $ and $\zeta$ = 7.

### b. What do you see from a comparison of the power spectrum?

*Note, the k-bins used forin each $n_{\rm grid}$ is different.
 Ideally, you should pass the k-edges array yourself for this exercise*

In [ ]:
ngrid_arr = np.array([64, 32, 16])
zeta_value = 7.0
log10Mmin_value = 8.0

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

ax_ps = axes[0]

for loopidx, ngrid_value in enumerate(ngrid_arr):

    print(
        f"log10Mmin = {log10Mmin_value}, "
        f"zeta = {zeta_value}, "
        f"ngrid = {ngrid_value}"
    )

    model_kbins, model_Delta_sq_21cm, model_QHII_mean, model_Delta_HI_arr, model_qi_arr, model_fcoll_mass_mean \
    = SCRIPT_model(
        log10Mmin_value,
        zeta_value,
        z=7,
        ngrid=ngrid_value
    )

    print(
        'mass averaged ionized fraction = '
        + '{:.2f}'.format(model_QHII_mean)
    )

    # ------------------------------------------
    # 21 cm power spectrum
    # ------------------------------------------
    ax_ps.plot(
        model_kbins,
        model_Delta_sq_21cm,
        lw=2,
        label=rf'$N_{{\rm grid}}={ngrid_value}$',
        marker='.',
        ls='--'
    )

    # ------------------------------------------
    # Delta_HI map slice
    # ------------------------------------------

    ngrid = model_Delta_HI_arr.shape[0]
    slice_index = ngrid // 2

    ax = axes[loopidx + 1]

    im = ax.imshow(
        model_Delta_HI_arr[:, :, slice_index],
        origin='lower',
        vmin=0,
        vmax=1,
        interpolation='bicubic',
    )

    ax.set_title(
        rf'$N_{{\rm grid}}={ngrid_value}$' '\n'
        rf'$\langle Q^M_{{\rm HII}}\rangle={model_QHII_mean:.2f}$'
    )

    ax.set_xticks([])
    ax.set_yticks([])

    print("-------")

# ------------------------------------------
# Power spectrum panel formatting
# ------------------------------------------

ax_ps.set_xscale('log')
ax_ps.set_yscale('log')

ax_ps.set_xlabel(
    r'$k\,[h\,{\rm cMpc}^{-1}]$'
)

ax_ps.set_ylabel(
    r'$\Delta^2_{21}(k)\,[{\rm mK}^2]$'
)

ax_ps.set_title(
    rf'$\zeta={zeta_value:.1f},\ '
    rf'\log_{{10}}(M_{{\rm min}}/M_\odot)={log10Mmin_value:.1f}$'
)

ax_ps.legend()

plt.tight_layout()
fig.subplots_adjust(bottom=0.18)

cax = fig.add_axes([0.35, 0.08, 0.45, 0.03])

cbar = fig.colorbar(
    im,
    cax=cax,
    orientation='horizontal'
)

cbar.set_label(
    r'$\Delta_{\rm HI}=x_{\rm HI}(1+\delta)$'
)

plt.savefig(
    './saved_figures/SCRIPT_output_vary_ngrid_maps.pdf'
)

plt.show()


## **Exercise 4 : Looking at 21 cm Fluctuations at Fixed Ionized Fraction**




In this exercise, we will compare different reionization models that produce approximately the same mean ionized fraction but differ in their source properties. This will help us understand how the morphology of ionized regions influences the observable 21 cm power spectrum.

### a. Generate ionization maps and 21 cm power spectra for the following combinations of $(\log_{10} M_{\rm min}, \zeta)$:

- $(\log_{10} M_{\rm min}, \zeta)$ = $(8.0, \, 8.0)$
- $(\log_{10} M_{\rm min}, \zeta)$ = $(9.0, \, 16.94)$
- $(\log_{10} M_{\rm min}, \zeta)$ = $(10.0, \, 60.52)$

Verify that all three models produce approximately the same mean ionized fraction.

### b. Compare the ionization maps and 21 cm power spectra for these set of models

- Do models with the same mean ionized fraction produce identical 21 \,cm power spectra?
- At which scales are the differences largest?
- How are the differences in the power spectra related to the morphology of the ionized regions seen in the maps?

In [ ]:
target_QHII_mean = 0.52

def get_zeta_for_target_QHII_mean(log10Mmin, target_QHII_mean):
    fcoll_arr = matter_fields.get_zeta_fcoll(1, log10Mmin)

    fcoll_mass_mean = np.mean(fcoll_arr * (1 + matter_fields.densitycontr_arr))

    print(f"target QHII_mean = {target_QHII_mean:.2f}")
    print(f"model fcoll_mass_mean = {fcoll_mass_mean:.2f}")
    zeta_req = target_QHII_mean / fcoll_mass_mean
    print(f"required zeta to achieve target QHII_mean = {zeta_req:.2f}")

log10Mmin_arr = np.array([8.0, 9.0, 10.0])

for log10Mmin_value in log10Mmin_arr:
    print(f"log10Mmin = {log10Mmin_value}")
    get_zeta_for_target_QHII_mean(log10Mmin_value, target_QHII_mean)
    print("-------")

In [ ]:
model_params = [
    (8.0, 8.0),
    (9.0, 16.94),
    (10.0, 60.52)
]



fig, axes = plt.subplots(1, 4,figsize=(24, 6))

ax_ps = axes[0]



for loopidx, (log10Mmin_value, zeta_value) in enumerate(model_params):

    print(
        f"log10Mmin = {log10Mmin_value}, "
        f"zeta = {zeta_value}"
    )

    model_kbins, model_Delta_sq_21cm, model_QHII_mean, model_Delta_HI_arr, model_qi_arr, model_fcoll_mass_mean = SCRIPT_model(
        log10Mmin_value,
        zeta_value
    )

    print(
        'mass averaged ionized fraction = '
        + '{:.2f}'.format(model_QHII_mean)
    )


    ax_ps.plot(
        model_kbins,
        model_Delta_sq_21cm,
        lw=2,
        label=(
            rf'$\log_{{10}}M_{{\min}}='
            rf'{log10Mmin_value:.0f},\ '
            rf'\zeta={zeta_value:.2f}$'
        )
    )

    # Central slice
    slice_index = model_Delta_HI_arr.shape[2] // 2

    ax = axes[loopidx + 1]

    im = ax.imshow(
        model_Delta_HI_arr[:, :, slice_index],
        origin='lower',
        vmin=0,
        vmax=1,
        interpolation='bicubic',
    )

    ax.set_title(
        rf'$\log_{{10}}M_{{\min}}={log10Mmin_value:.0f}$'
        '\n'
        rf'$\zeta={zeta_value:.2f}$'
        '\n'
        rf'$\langle Q^M_{{\rm HII}}\rangle='
        rf'{model_QHII_mean:.2f}$'
    )

    ax.set_xticks([])
    ax.set_yticks([])

    print("-------")

ax_ps.set_xscale('log')
ax_ps.set_yscale('log')

ax_ps.set_xlabel(
    r'$k\,[h\,{\rm cMpc}^{-1}]$'
)

ax_ps.set_ylabel(
    r'$\Delta^2_{21}(k)\,[{\rm mK}^2]$'
)



ax_ps.legend(fontsize=14)
plt.tight_layout()

# Leave room at the bottom
fig.subplots_adjust(bottom=0.18)

# [left, bottom, width, height]
cax = fig.add_axes([0.35, 0.08, 0.45, 0.03])

cbar = fig.colorbar(
    im,
    cax=cax,
    orientation='horizontal'
)

cbar.set_label(r'$\Delta_{\rm HI} = x_{\rm HI}(1+\delta)$')

plt.savefig(
    './saved_figures/SCRIPT_output_fixed_QHII.pdf'
)

plt.show()

## **Exercise 5: SCRIPT-ing the history of reionization**

In the previous exercises, we explored how the ionized fraction and the 21 cm signal depend on the properties of ionizing sources at a fixed redshift. In this exercise, we will instead follow the **time evolution of reionization** by constructing ionization maps over a sequence of simulation snapshots.

The supplied simulation snapshots span the redshift range $
15 \lesssim z \lesssim 5$, covering the entire Epoch of Reionization, from the appearance of the first ionized regions to the nearly fully ionized Universe.

For each snapshot, you will use the SCRIPT model to compute the ionization field,$x_{\rm HII}(\mathbf{x},z),$ and the corresponding neutral hydrogen overdensity field,

$$
\Delta_{\rm HI}(\mathbf{x},z)
=
\left[1-x_{\rm HII}(\mathbf{x},z)\right]
\left[1+\delta(\mathbf{x},z)\right].
$$

Your goal is to combine these maps into a **movie of cosmic reionization**, visualizing how ionized bubbles emerge, grow, and eventually overlap as the Universe evolves from $z\sim15$ to $z\sim5$. By comparing the evolving neutral hydrogen maps with the underlying matter density field, you will gain an intuitive understanding of the morphology and progress of cosmic reionization.



In [ ]:
ngrid = 32
log10Mmin = 8.0
zeta = 7

centre = 0.5 ## in units of box size

z_arr = np.array([])
QHI_mean_arr = np.array([])
Delta_HI_plot_arr = np.empty( shape=(0, ngrid, ngrid) )
Delta_DM_plot_arr = np.empty( shape=(0, ngrid, ngrid) )

zgrid = int(ngrid * centre)


loop = 0
snap = 0
delta_snap = 1
sigma_8=0.829,
ns=0.961,
omega_b=0.0482,

while(True):
    snap_str = '{:03d}'.format(snap)

    gadget_snap = os.path.join(work_path,
    "N128L128_2LPT",
    f"snap_{snap_str}"
    )
    outpath=  os.path.join(work_path,
    'script_saved_files')

    if (os.path.exists(gadget_snap)):
        print("snap = ", snap)
        gadget_data = script.default_simulation_data(gadget_snap, outpath, sigma_8=sigma_8, ns=ns, omega_b=omega_b)
        dens_fields = script.matter_fields(gadget_data, ngrid, outpath, overwrite_files=False)
        ionmap = script.ionization_map(dens_fields)

        fcoll_arr = dens_fields.get_fcoll_for_Mmin(log10Mmin)
        qi_arr = ionmap.get_qi(zeta * fcoll_arr)

        Delta_HI_arr = (1 - qi_arr) * (1 + dens_fields.densitycontr_arr)
        Delta_DM_arr = (1 + dens_fields.densitycontr_arr)


        box = gadget_data.box
        Delta_HI_plot_arr = np.concatenate((Delta_HI_plot_arr, [Delta_HI_arr[:,:,zgrid].T]), axis = 0)

        Delta_DM_plot_arr = np.concatenate((Delta_DM_plot_arr, [Delta_DM_arr[:,:,zgrid].T]), axis = 0)

        z_arr = np.append(z_arr, gadget_data.z)
        QHI_mean_arr = np.append(QHI_mean_arr, np.mean(Delta_HI_arr))



    else: ## when we do not find the next snapshot
        break

    loop += 1
    snap += delta_snap

#print ("k bin values: ", k_bins)

num_maps = loop

In [ ]:
# Sort by redshift in case snapshots are not read in order
sort_idx = np.argsort(z_arr)

z_plot = z_arr[sort_idx]
QHII_plot = QHI_mean_arr[sort_idx]

fig, ax = plt.subplots(figsize=(8, 6))

ax.plot(
    z_plot,
    QHII_plot,
    '-o',
    lw=2,
    ms=6
)

ax.set_xlabel(r'Redshift $z$')
ax.set_ylabel(r'$\langle Q_{\rm HII}^{M}\rangle$(z)')

ax.set_title(
    rf'$\log_{{10}}M_{{\min}}={log10Mmin:.1f},\ '
    rf'\zeta={zeta:.1f}$'
)

ax.grid(alpha=0.3)



plt.tight_layout()
plt.show()

In [ ]:
from matplotlib.animation import ArtistAnimation

print('Preparing the movie now...')

# ==========================================================
# Figure with two panels:
# Left  : Dark matter overdensity
# Right : Neutral hydrogen field
# ==========================================================

fig, (ax_dm, ax_hi) = plt.subplots(
    1, 2,
    figsize=(12, 5),
    constrained_layout=True
)

# Colour scales
vmin_dm = np.min(Delta_DM_plot_arr)
vmax_dm = np.max(Delta_DM_plot_arr)

vmin_hi = np.min(Delta_HI_plot_arr)
vmax_hi = np.max(Delta_HI_plot_arr)

# Initial images
im_dm = ax_dm.imshow(
    Delta_DM_plot_arr[0],
    origin='lower',
    extent=[0, box, 0, box],
    interpolation='bicubic',
    cmap='hot',
    vmin=vmin_dm,
    vmax=vmax_dm
)

im_hi = ax_hi.imshow(
    Delta_HI_plot_arr[0],
    origin='lower',
    extent=[0, box, 0, box],
    interpolation='bicubic',
    cmap='viridis',
    vmin=vmin_hi,
    vmax=vmax_hi
)

# Colourbars
cbar_dm = fig.colorbar(
    im_dm,
    ax=ax_dm,
    fraction=0.05,
    pad=0.03,
    shrink=0.8
)

cbar_hi = fig.colorbar(
    im_hi,
    ax=ax_hi,
    fraction=0.05,
    pad=0.03,
    shrink=0.8
)
cbar_dm.set_label(
    r'$\Delta_{\rm DM}=1+\delta$'
)

cbar_hi.set_label(
    r'$\Delta_{\rm HI}=x_{\rm HI}(1+\delta)$'
)

# Axis labels
for ax in [ax_dm, ax_hi]:

    ax.set_xlabel(
        r'$x\,[h^{-1}{\rm cMpc}]$'
    )

    ax.set_ylabel(
        r'$y\,[h^{-1}{\rm cMpc}]$'
    )

# Titles
ax_dm.set_title('Dark Matter ')
ax_hi.set_title('Neutral Hydrogen')

# ==========================================================
# Build movie frames
# ==========================================================

ims = []

for frame in range(num_maps):
    # DM map
    image_dm = ax_dm.imshow(
        Delta_DM_plot_arr[frame],
        origin='lower',
        extent=[0, box, 0, box],
        interpolation='bicubic',
        cmap='hot',
        vmin=vmin_dm,
        vmax=vmax_dm
    )

    # HI map
    image_hi = ax_hi.imshow(
        Delta_HI_plot_arr[frame],
        origin='lower',
        extent=[0, box, 0, box],
        interpolation='bicubic',
        cmap='viridis',
        vmin=vmin_hi,
        vmax=vmax_hi
    )

    # Labels
    text_dm = ax_dm.text(
        0.05 * box,
        0.92 * box,
        rf'$z={z_arr[frame]:.2f}$',
        bbox=dict(
            facecolor='white',
            alpha=0.7
        )
    )

    text_hi = ax_hi.text(
        0.05 * box,
        0.92 * box,
        rf'$\langle Q_{{\rm HII}}\rangle='
        rf'{QHI_mean_arr[frame]:.2f}$',
        bbox=dict(
            facecolor='white',
            alpha=0.7
        )
    )

    ims.append([
        image_dm,
        image_hi,
        text_dm,
        text_hi
    ])

# ==========================================================
# Create and save animation
# ==========================================================

frame_rate = 2

animation = ArtistAnimation(
    fig,
    ims,
    interval=1000 / frame_rate
)

animation.save(
    './saved_figures/DM_HI_movie.gif',
    dpi=256
)
print("Animation is saved")

from IPython.display import HTML
plt.close(fig)      # Prevent the static figure from rendering

HTML(animation.to_jshtml())
